# Análisis de Datos · Semana 7
## Selección anidada y operadores lógicos

**TIA502 · Facultad de Empresariales · Profesor David Escobar-Castillejos**

La sesión pasada fue una condición a la vez. Esta es cómo se juntan, y cuándo conviene anidar en
lugar de combinar.

Al terminar este cuaderno vas a poder:

1. Usar los tres operadores lógicos: `and` exige las dos, `or` se conforma con una, `not`
   invierte.
2. Leer una tabla de verdad y predecir el resultado de una condición compuesta sin ejecutarla.
3. Preguntar por pertenencia con `in` y `not in`, en lugar de encadenar comparaciones con `or`.
4. Distinguir `is` de `==`.
5. Decidir cuándo anidar, y reconocer el anidado que en realidad era un `and`.

### Cómo se usa este cuaderno

Ejecuta las celdas en orden. Cuatro fallan a propósito o dan un resultado inesperado a propósito,
y llevan un comentario que lo dice.

La regla práctica del final vale más que toda la teoría de arriba: **si las dos ramas del anidado
hacen lo mismo, era un `and` disfrazado.**

---
# Bloque 1 · Combinar condiciones

Tres operadores, y con ellos se arma cualquier regla por complicada que suene al decirla en voz
alta.

| Operador | Qué exige | Ejemplo |
|---|---|---|
| `and` | Que las dos condiciones sean verdaderas | `conversion >= 0.03 and clics > 1000` |
| `or` | Que al menos una sea verdadera | `canal == "Meta" or canal == "Google"` |
| `not` | Invierte el resultado de la condición | `not campana_activa` |

## La tabla de verdad, generada

| A | B | `A and B` | `A or B` |
|---|---|---|---|
| `True` | `True` | `True` | `True` |
| `True` | `False` | `False` | `True` |
| `False` | `True` | `False` | `True` |
| `False` | `False` | `False` | `False` |

En lugar de creérmela, constrúyela.

In [ ]:
print(f"{'A':<7}{'B':<7}{'A and B':<10}{'A or B':<10}{'not A':<7}")
print("-" * 41)

for a in [True, False]:
    for b in [True, False]:
        print(f"{str(a):<7}{str(b):<7}{str(a and b):<10}{str(a or b):<10}{str(not a):<7}")

Cuatro renglones y ahí está todo. `and` solo es verdadero en el primero; `or` es falso solo en el
último.

## Una regla con dos condiciones

In [ ]:
conversion = 0.0342
clics = 5074
canal = "Instagram"

if conversion >= 0.03 and clics > 1000:
    print("La campaña califica para más presupuesto.")
else:
    print("La campaña se mantiene igual.")

Con `and`, si una sola falla, la regla completa falla. No hay término medio.

**Por qué la segunda condición.** Una conversión alta sobre cien clics no significa nada: puede ser
casualidad. El volumen filtra el ruido, y por eso la política pide las dos cosas.

Cámbiale el `and` por un `or` y mira a quién aprueba.

In [ ]:
CANDIDATAS = [
    ("Instagram", 0.0342, 5074),
    ("LinkedIn", 0.0205, 640),
    ("Boletín", 0.0810, 62),      # conversión altísima, casi sin volumen
    ("Display", 0.0021, 88400),   # volumen enorme, conversión pésima
]

print(f"{'Canal':<12}{'Conv.':>8}{'Clics':>9}   con and        con or")
print("-" * 58)

for canal, conv, clics in CANDIDATAS:
    con_and = "aprueba" if (conv >= 0.03 and clics > 1000) else "no"
    con_or = "aprueba" if (conv >= 0.03 or clics > 1000) else "no"
    print(f"{canal:<12}{conv:>8.2%}{clics:>9,}   {con_and:<14} {con_or}")

Con `or`, Display aprueba: ochenta y ocho mil clics y una conversión del 0.21 %. Es la campaña
que más dinero quema del grupo y la regla la premia.

Ese es el costo de cambiar una palabra.

## Evaluación corta

Si la primera condición de un `and` es falsa, Python **ni siquiera lee la segunda**. Ahorra
trabajo y, más importante, evita errores.

In [ ]:
def revisar(nombre):
    """Dice en voz alta que fue evaluada, para que se vea cuándo corre."""
    print(f"  (evaluando {nombre})")
    return True


print("Con la primera falsa:")
resultado = False and revisar("segunda")
print("Resultado:", resultado)

print()
print("Con la primera verdadera:")
resultado = True and revisar("segunda")
print("Resultado:", resultado)

En el primer caso `revisar` nunca corrió. Eso no es una curiosidad: es lo que permite escribir
condiciones que serían un error al revés.

In [ ]:
clics_reportados = 0

# El orden correcto: primero se revisa que no sea cero, después se divide.
if clics_reportados > 0 and 38500 / clics_reportados < 10:
    print("Costo por clic aceptable")
else:
    print("Sin clics suficientes para evaluar")

In [ ]:
# FALLA A PROPÓSITO. El mismo par de condiciones, en el orden equivocado.
try:
    if 38500 / clics_reportados < 10 and clics_reportados > 0:
        print("Costo por clic aceptable")
except ZeroDivisionError as e:
    print("ZeroDivisionError:", e)

La misma regla, dos comportamientos. Con la guarda primero, la división nunca ocurre; al revés,
truena antes de llegar a la guarda.

**Cuando una condición protege a la otra, va primero.** No es estilo, es lo que hace que el
programa corra.

## `not`

Invierte. Se usa poco y cuando se usa, conviene que la variable ya se lea como una afirmación.

In [ ]:
campana_activa = False

print("not campana_activa:", not campana_activa)

if not campana_activa:
    print("La campaña está pausada, no hay nada que evaluar.")

`if not campana_activa:` se lee casi como en español. Comparar con `if campana_activa == False:`
funciona igual y se lee peor.

---
# Bloque 2 · Pertenencia e identidad

Cuatro operadores más, y dos de ellos te van a ahorrar escribir la misma comparación cinco veces.

| Operador | Pregunta | Ejemplo | Resultado |
|---|---|---|---|
| `in` | ¿Está dentro? | `canal in ["Meta", "Google"]` | `False` |
| `not in` | ¿Está fuera? | `canal not in ["TikTok"]` | `True` |
| `is` | ¿Es el mismo objeto? | `costo is None` | `True` |
| `is not` | ¿Es otro objeto? | `costo is not None` | `False` |

## Cinco comparaciones, o una

Así se ve encadenando con `or`:

```python
if (canal == "Meta" or
    canal == "Google" or
    canal == "Instagram" or
    canal == "TikTok"):
    print("Canal digital")
```

Y así con `in`:

In [ ]:
DIGITALES = ["Meta", "Google", "Instagram", "TikTok"]

canal = "Instagram"

if canal in DIGITALES:
    print("Canal digital")
else:
    print("Otro canal")

La segunda se lee de un vistazo y, sobre todo, **la lista se puede cambiar sin tocar la
condición**. Agregar un canal es agregar un elemento.

In [ ]:
DIGITALES.append("LinkedIn")

for canal in ["Instagram", "LinkedIn", "Radio", "Espectacular"]:
    print(f"{canal:<14} {'digital' if canal in DIGITALES else 'tradicional'}")

`in` también funciona sobre texto, y ahí pregunta si una cadena está contenida en otra.

In [ ]:
puesto = "Sales analyst"

print('"analyst" in puesto :', "analyst" in puesto)
print('"Analyst" in puesto :', "Analyst" in puesto, "<- distingue mayúsculas")
print('"analyst" in puesto.lower() :', "analyst" in puesto.lower())

Ese es el antecesor directo de `.str.contains("manager", case=False)` de la semana 15.2, aplicado
a un valor en vez de a una columna.

## `is` no es `==`

El que más se confunde. `==` pregunta si valen lo mismo; `is` pregunta si son **el mismo objeto**.

In [ ]:
a = [1, 2, 3]
b = [1, 2, 3]

print("a == b :", a == b, "<- valen lo mismo")
print("a is b :", a is b, "<- y no son el mismo objeto")

c = a
print("c is a :", c is a, "<- c es otro nombre para el mismo objeto")

Dos listas con el mismo contenido son iguales y no son la misma. Es la diferencia entre dos hojas
con los mismos datos y dos pestañas que apuntan al mismo archivo.

**Para números y texto, usa siempre `==`.** `is` con valores da resultados que dependen de detalles
internos de Python y no se pueden predecir.

In [ ]:
# FALLA A PROPÓSITO, y de la peor forma: a veces funciona.
x = 256
y = 256
print("256 is 256 :", x is y)

x = 1000
y = 1000
print("1000 is 1000 :", x is y, "<- el mismo código, otro resultado")

print()
print("Con == siempre es predecible:", 1000 == 1000)

El mismo código con otro número da otro resultado, porque Python guarda los enteros chicos en una
tabla y los reutiliza. Nada de eso es algo en lo que debas apoyarte.

`is` tiene un uso correcto y es este:

In [ ]:
costo_por_clic = None

print("costo is None     :", costo_por_clic is None)
print("costo is not None :", costo_por_clic is not None)

costo_por_clic = 7.59
print("Ya medido, is not None:", costo_por_clic is not None)

`is None` y `is not None` son la forma correcta de preguntar por ausencia de dato, porque `None`
es un objeto único en todo el programa.

---
# Bloque 3 · Anidar una decisión

Una decisión dentro de otra. A veces es lo correcto, y a veces es un `and` escrito de la forma más
larga posible.

## Un anidado que sí gana algo

In [ ]:
campana_activa = False
conversion = 0.061

if campana_activa:
    if conversion >= 0.05:
        accion = "Subir presupuesto"
    elif conversion >= 0.03:
        accion = "Mantener"
    else:
        accion = "Pausar y revisar"
else:
    accion = "Reactivar antes de evaluar"

print(accion)

Aquí el anidado gana algo real, por tres razones.

**La primera pregunta decide si sigues.** Si la campaña está apagada, su conversión no significa
nada: son datos de cuando estaba prendida. Preguntar por ella sería un error de negocio.

**Cada rama interna hace algo distinto.** Tres salidas, no dos iguales.

**El `else` de afuera** cubre el caso apagado completo, sin repetir las tres categorías.

La traza con una campaña apagada y conversión del 6 %:

| Paso | Condición | Resultado | `accion` |
|---|---|---|---|
| 1 | `campana_activa` | `False` | – |
| 2 | Todo el bloque interno | No se evalúa | – |
| 3 | `else` de afuera | Se ejecuta | `Reactivar antes de evaluar` |

La conversión del 6 % nunca se mira. El anidado la protege de una decisión que no tendría sentido
tomar.

## El anidado que era un `and`

Ahora el caso contrario. Léelo y busca qué le sobra.

In [ ]:
def aprobar_anidado(conversion, clics):
    """Tres niveles de sangría para una sola pregunta."""
    if conversion >= 0.03:
        if clics > 1000:
            return "aprueba"
        else:
            return "no aprueba"
    else:
        return "no aprueba"


def aprobar_combinado(conversion, clics):
    """Lo mismo, en una línea."""
    return "aprueba" if (conversion >= 0.03 and clics > 1000) else "no aprueba"


for conv, clics in [(0.0342, 5074), (0.081, 62), (0.0021, 88400), (0.02, 500)]:
    a = aprobar_anidado(conv, clics)
    b = aprobar_combinado(conv, clics)
    print(f"{conv:>7.2%}{clics:>8,}   {a:<12}{b:<12}{'iguales' if a == b else 'DISTINTOS'}"),

Idénticos en los cuatro casos, y uno ocupa nueve líneas y el otro una.

**La prueba: si las dos ramas internas hacen lo mismo, era un `and`.** En `aprobar_anidado`, el
`else` de adentro y el de afuera devuelven exactamente lo mismo, y eso es la señal.

Es la revisión más rentable que le puedes hacer a tu propio código. Convierte cuatro niveles de
sangría en una línea legible.

## La regla, dicha completa

Se anida cuando:

1. La segunda pregunta **solo tiene sentido** si la primera se cumplió.
2. Cada rama hace **algo distinto**.

Si las dos ramas internas terminan haciendo lo mismo, o si la segunda pregunta se puede hacer
siempre, entonces no era un anidado: era una sola condición unida con `and`.

---
## Cuatro trampas de las condiciones compuestas

### Escribir `and` cuando querías `or`

Léela en voz alta. "Las dos" es `and`, "cualquiera de las dos" es `or`. La mitad de los errores se
atrapan así.

### Comparar contra dos valores de golpe

**Predice antes de correr.** ¿Qué imprime, si el canal es Instagram?

- **A.** `No coincide`, porque no es ninguno de los dos.
- **B.** `Coincide`, porque un texto no vacío se evalúa como verdadero.
- **C.** Un error, porque falta una comparación.
- **D.** `Coincide`, porque Python compara con los dos.

In [ ]:
# FALLA A PROPÓSITO, sin lanzar nada. Esta condición no hace lo que parece.
canal = "Instagram"

if canal == "Meta" or "Google":
    print("Coincide")
else:
    print("No coincide")

La respuesta es **B**, y es de las peores trampas del lenguaje.

Python lee eso como `(canal == "Meta") or ("Google")`. La primera parte es falsa, así que evalúa la
segunda: `"Google"` a secas, un texto no vacío, que cuenta como verdadero.

La condición es verdadera **siempre**, para cualquier canal.

In [ ]:
for canal in ["Instagram", "Meta", "Radio", "cualquier cosa"]:
    resultado = canal == "Meta" or "Google"
    print(f"{canal:<16} -> {resultado!r}")

Ni siquiera devuelve `True`: devuelve el texto `"Google"`, que en un `if` cuenta como verdadero.

Las dos formas correctas:

In [ ]:
canal = "Instagram"

print("Repitiendo la variable:", canal == "Meta" or canal == "Google")
print("Con in:                ", canal in ("Meta", "Google"))

### Usar `is` para comparar valores

Ya lo viste arriba. Para números y texto, siempre `==`.

### Anidar sin necesidad

Tres niveles de sangría casi siempre son dos condiciones unidas con `and` y una rama que sobra.

## Todo junto: una política real

In [ ]:
DIGITALES = ["Meta", "Google", "Instagram", "TikTok", "LinkedIn"]

def decidir_presupuesto(canal, activa, conversion, clics):
    """Política completa, con los tres operadores lógicos y pertenencia."""
    if not activa:
        return "Reactivar antes de evaluar"

    if canal not in DIGITALES:
        return "Fuera de política, revisar a mano"

    if conversion >= 0.05 and clics > 1000:
        return "Subir presupuesto"
    elif conversion >= 0.03 or clics > 50000:
        return "Mantener"
    else:
        return "Pausar y revisar"


CASOS = [
    ("Instagram", True, 0.061, 5074),
    ("Instagram", False, 0.061, 5074),
    ("Radio", True, 0.061, 5074),
    ("Display", True, 0.0021, 88400),
    ("LinkedIn", True, 0.0205, 640),
]

for canal, activa, conv, clics in CASOS:
    estado = "activa" if activa else "pausada"
    print(f"{canal:<11}{estado:<10}{conv:>7.2%}{clics:>8,}   {decidir_presupuesto(canal, activa, conv, clics)}")

Tres guardas al principio y una decisión al final. Ninguna sangría pasa de dos niveles, y la
función se lee de arriba abajo como una lista de reglas.

Ese patrón, sacar los casos especiales primero y dejar la lógica principal al final, es lo que
evita el anidado profundo casi siempre.

---
# Ejercicios

Las soluciones están hasta abajo del cuaderno.

## Lógicos

### Ejercicio 1 · La tabla de verdad de `not` y de la combinación

Genera con un ciclo la tabla de verdad de `A and not B` y de `not (A or B)`. Cuatro renglones cada
una.

Después contesta en un comentario: ¿alguna de las dos da lo mismo que `not A or not B`?

### Ejercicio 2 · Leerla en voz alta

Para cada una de estas tres reglas, escribe la condición en Python y la frase en español que la
describe:

1. Aprobar si el cliente tiene más de dos años **y** su saldo es menor a 10 000.
2. Alertar si el pedido pasa de 100 000 **o** el cliente es nuevo.
3. Rechazar si **no** está en la lista de proveedores autorizados.

### Ejercicio 3 · La guarda que protege

Escribe una condición que calcule el costo por clic solo si hay clics, usando evaluación corta.
Pruébala con cero clics y con clics de verdad.

Después escríbela al revés y comprueba que truena.

## Pertenencia

### Ejercicio 4 · De cinco `or` a un `in`

Escribe una condición con cuatro `or` que revise si un mes es del último trimestre, y después la
misma con `in`. Pruébalas con seis meses distintos y comprueba que dan lo mismo.

### Ejercicio 5 · Buscar dentro de un texto

Con esta lista de puestos, imprime los que contengan la palabra "manager" sin importar mayúsculas,
y por separado los que contengan "analyst".

```python
PUESTOS = ["Sales analyst", "Brand Manager", "people manager",
           "Financial Analyst", "Recruiter", "Operations Manager"]
```

### Ejercicio 6 · `is` contra `==`

Crea dos diccionarios con el mismo contenido y compáralos con `==` y con `is`. Después asigna uno
al otro y vuelve a comparar.

Explica en un comentario en qué caso `is` sería la pregunta correcta.

## Anidar

### Ejercicio 7 · Colapsar un anidado

Este código tiene tres niveles de sangría y dos ramas que hacen lo mismo. Reescríbelo en una sola
condición.

```python
def puede_enviar(peso, destino, pagado):
    if pagado:
        if peso <= 20:
            if destino != "internacional":
                return "enviar"
            else:
                return "no enviar"
        else:
            return "no enviar"
    else:
        return "no enviar"
```

Comprueba con ocho combinaciones que las dos versiones dan lo mismo.

### Ejercicio 8 · Una política que necesita dos condiciones

Escribe una regla de tu área que dependa de al menos dos datos: aprobar un crédito por ingreso y
antigüedad, o priorizar un pedido por monto y por cliente. Que use `and`, `or` e `in` al menos una
vez cada uno.

Máximo dos niveles de sangría. Si necesitas tres, colapsa con `and`.

La prueba: léela en voz alta a un compañero. Si tiene que preguntar "¿y o o?", la condición está
mal escrita.

---
## Tres ideas para llevarse

**`and` exige las dos, `or` se conforma con una.** Leer la condición en voz alta atrapa la mitad de
los errores antes de correr el programa.

**`in` reemplaza una fila de `or`.** Y deja que la lista de valores válidos cambie sin tocar una
sola línea de la condición.

**Si las dos ramas hacen lo mismo, era un `and`.** La revisión más rentable que le puedes hacer a
tu código, y convierte cuatro niveles de sangría en uno.

La siguiente sesión es repetición, y el primer examen parcial.

---
# Soluciones

### Ejercicio 1

```python
print(f"{'A':<7}{'B':<7}{'A and not B':<14}{'not (A or B)':<15}{'not A or not B':<15}")
for a in [True, False]:
    for b in [True, False]:
        print(f"{str(a):<7}{str(b):<7}{str(a and not b):<14}"
              f"{str(not (a or b)):<15}{str(not a or not b):<15}")

# not (A or B) no coincide con not A or not B. La que sí coincide con
# not A or not B es not (A and B). Es una de las leyes de De Morgan: al negar
# una combinación, el and se vuelve or y cada parte se niega.
```

Vale la pena reconocer esa ley aunque no se llame por su nombre. Aparece cada vez que alguien
intenta negar una condición compuesta y lo hace a la mitad.

### Ejercicio 2

```python
antiguedad_anios = 3
saldo = 8500
monto_pedido = 128000
cliente_nuevo = False
AUTORIZADOS = ["Insumos SA", "Papelera del Norte", "Log Express"]
proveedor = "Otro Proveedor"

# 1. "Más de dos años Y saldo menor a diez mil"
print("Aprobar:", antiguedad_anios > 2 and saldo < 10000)

# 2. "Pedido de más de cien mil O cliente nuevo"
print("Alertar:", monto_pedido > 100000 or cliente_nuevo)

# 3. "NO está en la lista de autorizados"
print("Rechazar:", proveedor not in AUTORIZADOS)
```

La tercera se puede escribir `not (proveedor in AUTORIZADOS)` y da lo mismo. `not in` existe
precisamente porque se lee mejor.

### Ejercicio 3

```python
inversion = 38500

for clics in [0, 5074]:
    if clics > 0 and inversion / clics < 10:
        print(f"{clics:>6} clics -> costo por clic aceptable")
    else:
        print(f"{clics:>6} clics -> no evaluable o costo alto")

# Al revés truena:
try:
    clics = 0
    if inversion / clics < 10 and clics > 0:
        print("nunca llega aquí")
except ZeroDivisionError as e:
    print("ZeroDivisionError:", e)
```

La versión correcta no necesita un `if` extra ni un `try`. La guarda dentro del mismo `and` hace
todo el trabajo, y eso solo funciona por la evaluación corta.

### Ejercicio 4

```python
ULTIMO_TRIMESTRE = ["oct", "nov", "dic"]

for mes in ["ene", "jun", "sep", "oct", "nov", "dic"]:
    con_or = mes == "oct" or mes == "nov" or mes == "dic"
    con_in = mes in ULTIMO_TRIMESTRE
    print(f"{mes}   or: {str(con_or):<6} in: {str(con_in):<6} {'ok' if con_or == con_in else 'DIFIEREN'}")
```

Las dos dan lo mismo siempre. La de `in` gana cuando la política cambia: si el trimestre ahora
empieza en septiembre, se agrega un elemento a la lista y ninguna condición se toca.

### Ejercicio 5

```python
PUESTOS = ["Sales analyst", "Brand Manager", "people manager",
           "Financial Analyst", "Recruiter", "Operations Manager"]

print("Gerencias:")
for p in PUESTOS:
    if "manager" in p.lower():
        print("  ", p)

print("Analistas:")
for p in PUESTOS:
    if "analyst" in p.lower():
        print("  ", p)
```

El `.lower()` va sobre el puesto, no sobre la palabra buscada. Es el orden que importa: normalizas
el dato y comparas contra un valor que ya escribiste en minúsculas.

### Ejercicio 6

```python
uno = {"canal": "Instagram", "clics": 5074}
dos = {"canal": "Instagram", "clics": 5074}

print("uno == dos :", uno == dos)
print("uno is dos :", uno is dos)

tres = uno
print("tres is uno:", tres is uno)

tres["clics"] = 9999
print("uno después de tocar tres:", uno)

# is sería la pregunta correcta cuando lo que quieres saber es si dos nombres
# apuntan al mismo objeto, porque entonces modificar uno modifica al otro. La
# última línea lo demuestra: tocar tres cambió uno, y eso solo pasa cuando is
# da verdadero.
```

Ese comportamiento es la razón de fondo por la que `.copy()` existe en pandas, y por la que la
semana 15.2 empieza haciendo `ventas.copy()` antes de la demostración.

### Ejercicio 7

```python
def puede_enviar(peso, destino, pagado):
    return "enviar" if (pagado and peso <= 20 and destino != "internacional") else "no enviar"


CASOS = [(15, "nacional", True), (15, "nacional", False),
         (25, "nacional", True), (25, "nacional", False),
         (15, "internacional", True), (15, "internacional", False),
         (25, "internacional", True), (20, "local", True)]

for peso, destino, pagado in CASOS:
    print(f"{peso:>3} kg {destino:<15} {'pagado' if pagado else 'sin pagar':<10} -> {puede_enviar(peso, destino, pagado)}")
```

Nueve líneas y tres niveles de sangría se volvieron una. La señal estaba a la vista: los tres
`else` devolvían exactamente lo mismo.

### Ejercicio 8

No hay solución publicada porque la política es distinta para cada quien. Se califica sobre cuatro
cosas: que use los tres operadores, que la sangría no pase de dos niveles, que los cuatro casos de
la tabla de verdad estén probados, y que la condición se pueda leer en voz alta sin ambigüedad.